In [3]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 20.0 MB/s eta 0:00:00


In [8]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [9]:
import os
from pyngrok import ngrok

In [7]:
ngrok.kill()

In [10]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://spending-chastise-bullion.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://spending-chastise-bullion.ngrok-free.dev


True

In [11]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [12]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [13]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology, 簡稱明新科大或MHUST）是一所位於**臺灣新竹縣新豐鄉**的私立科技大學。

以下是明新科技大學的簡要介紹：

1.  **創校歷史與定位：**
    *   **創立年份：** 1966年，前身為「明新工業專科學校」。
    *   **改制科技大學：** 於2002年正式改制為明新科技大學。
    *   **辦學宗旨：** 以「理論與實務並重，培育術德兼備之專業人才」為辦學理念，強調實務技能的培養，與產業接軌。

2.  **地理位置優勢：**
    *   學校鄰近**新竹科學園區**及多個工業區，這使得明新科大在產學合作、學生實習與畢業就業方面具有得天獨厚的優勢。許多畢業生能直接進入竹科相關產業工作。

3.  **學院與系所：**
    *   明新科大設有多元學院，涵蓋工程、管理、設計與服務產業等領域，以因應不同產業的人才需求。主要學院包括：
        *   **工程學院：** 培育機械、電機、電子、資工等工程技術人才。
        *   **管理學院：** 涵蓋企業管理、行銷、資訊管理等領域。
        *   **服務事業學院：** 發展旅館管理、餐飲管理、幼兒保育、休閒事業等。
        *   **設計學院：** 培養多媒體與遊戲設計、時尚造型設計等創意設計人才。

4.  **教學特色：**
    *   **實務導向：** 強調動手實作能力，設有現代化的實驗室與實習工廠，提供學生充足的實務操作機會。
    *   **產學合作：** 與企業界建立緊密關係，推動建教合作、校外實習、專題研究等，讓學生在學期間就能接觸產業實務。
    *   **就業競爭力：** 學校課程設計貼近業界需求，致力於培養學生成為具備即戰力的專業人才，畢業生就業率表現良好。
    *   **國際化：** 積極推動國際交流與合作，提供學生海外研修、實習機會，培養學生的國際視野。

5.  **校園環境：**
    *   校園佔地廣闊，擁有完善的教學設施、圖書館、體育場館、學生活動中心等，提供學生良好的學習與生活環境。

總體而言，明新科技大學是一所深耕在地、放眼國際，以產業需求為導向，致力於培養高素質專業技術人

In [18]:
result2 = stateful_query("校長是誰？")
print(result2)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 38.922157373s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)